In [47]:
import pandas as pd
import numpy as np

crsp = pd.read_csv("../data/raw/crsp_monthly_raw.csv")

crsp.head()

,PERMNO,HdrCUSIP,PrimaryExch,ShareType,Ticker,PERMCO,SICCD,MthCalDt,MthPrc,MthRet,MthVol,ShrOut,DisFacPr,DisFacShr
0,10001,36720410,A,NS,EGAS,7953,4925,2010-01-29,10.0600,-0.019063,310400.0,4361,0.0,0.0
1,10001,36720410,A,NS,EGAS,7953,4925,2010-02-26,10.0084,-0.000652,151000.0,4361,0.0,0.0
2,10001,36720410,A,NS,EGAS,7953,4925,2010-03-31,10.1700,0.020719,228300.0,4361,0.0,0.0
3,10001,36720410,A,NS,EGAS,7953,4925,2010-04-30,11.3900,0.124568,335000.0,6070,0.0,0.0
4,10001,36720410,A,NS,EGAS,7953,4925,2010-05-28,11.4000,0.005155,345100.0,6071,0.0,0.0


In [48]:
crsp.columns = crsp.columns.str.lower()
crsp.head()

,permno,hdrcusip,primaryexch,sharetype,ticker,permco,siccd,mthcaldt,mthprc,mthret,mthvol,shrout,disfacpr,disfacshr
0,10001,36720410,A,NS,EGAS,7953,4925,2010-01-29,10.0600,-0.019063,310400.0,4361,0.0,0.0
1,10001,36720410,A,NS,EGAS,7953,4925,2010-02-26,10.0084,-0.000652,151000.0,4361,0.0,0.0
2,10001,36720410,A,NS,EGAS,7953,4925,2010-03-31,10.1700,0.020719,228300.0,4361,0.0,0.0
3,10001,36720410,A,NS,EGAS,7953,4925,2010-04-30,11.3900,0.124568,335000.0,6070,0.0,0.0
4,10001,36720410,A,NS,EGAS,7953,4925,2010-05-28,11.4000,0.005155,345100.0,6071,0.0,0.0


In [49]:
crsp["date"] = pd.to_datetime(crsp["mthcaldt"])

crsp["ret"] = pd.to_numeric(crsp["mthret"], errors="coerce")
crsp["price"] = pd.to_numeric(crsp["mthprc"], errors="coerce").abs()
crsp["shrout"] = pd.to_numeric(crsp["shrout"], errors="coerce")

In [50]:
crsp["me"] = crsp["price"] * crsp["shrout"]

In [51]:
crsp = crsp[[
    "permno",
    "date",
    "ret",
    "price",
    "shrout",
    "me"
]].copy()

In [52]:
crsp = crsp.dropna(subset=["ret", "price", "me"])
crsp = crsp[(crsp["price"] > 0) & (crsp["me"] > 0)]

In [53]:
crsp = crsp.sort_values(["permno", "date"])
crsp.head()

,permno,date,ret,price,shrout,me
0,10001,2010-01-29,-0.019063,10.0600,4361,43871.6600
1,10001,2010-02-26,-0.000652,10.0084,4361,43646.6324
2,10001,2010-03-31,0.020719,10.1700,4361,44351.3700
3,10001,2010-04-30,0.124568,11.3900,6070,69137.3000
4,10001,2010-05-28,0.005155,11.4000,6071,69209.4000


In [54]:
crsp.to_csv("../data/processed/crsp_clean.csv", index=False)

In [55]:
crsp["gross_ret"] = 1 + crsp["ret"]

crsp["momentum"] = (
    crsp.groupby("permno")["gross_ret"]
    .rolling(window=11)
    .apply(np.prod, raw=True)
    .reset_index(level=0, drop=True)
    - 1
)

crsp["momentum"] = crsp.groupby("permno")["momentum"].shift(2)

In [56]:
crsp[["permno", "date", "ret", "momentum"]].head(20)

,permno,date,ret,momentum
0,10001,2010-01-29,-0.019063,NaN
1,10001,2010-02-26,-0.000652,NaN
2,10001,2010-03-31,0.020719,NaN
3,10001,2010-04-30,0.124568,NaN
4,10001,2010-05-28,0.005155,NaN
5,10001,2010-06-30,-0.043732,NaN
6,10001,2010-07-30,0.083605,NaN
7,10001,2010-08-31,-0.111716,NaN
8,10001,2010-09-30,0.076567,NaN
9,10001,2010-10-29,0.032909,NaN
